# Frozen TFM tokenizer → ZuCo sentiment

This notebook tests whether the official pretrained TFM tokenizer transfers to ZuCo natural-reading EEG. It extracts frozen tokens, checks for codebook collapse, and evaluates a small token-histogram classifier against shuffled EEG and majority controls.

Use a **GPU** Colab runtime. Raw data, checkpoints, caches, and results remain in Colab/Google Drive and are never committed to GitHub.

In [ ]:
# 1) Fetch this small codebase and install tokenizer-only Colab dependencies.
from pathlib import Path
import os, subprocess, sys

PROJECT_URL = "https://github.com/parmisbathayan/EEGTokenizer.git"
PROJECT_ROOT = Path("/content/EEGTokenizer")

def run(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)

if not PROJECT_ROOT.exists():
    run(["git", "clone", "--depth", "1", PROJECT_URL, str(PROJECT_ROOT)])
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "tfm/requirements-colab.txt")])
os.chdir(PROJECT_ROOT / "tfm")
print("Working directory:", Path.cwd())

In [ ]:
# 2) Mount Drive and edit only these paths if your layout differs.
from google.colab import drive
drive.mount("/content/drive")

THESIS_ROOT = Path("/content/drive/MyDrive/Thesis")
DATA_ROOT = THESIS_ROOT / "Data"
CACHE_ROOT = THESIS_ROOT / "CachedArtifacts/eeg_tokenizer/tfm"
RESULTS_ROOT = THESIS_ROOT / "Results/eeg_tokenizer/tfm"

RAW_DIR = DATA_ROOT / "zuco_og_raw"
LABELS_CSV = DATA_ROOT / "zuco_sentiment_labels_task1_fixed.csv"
TOKEN_CACHE = CACHE_ROOT / "tokens_v1"
RESULTS_DIR = RESULTS_ROOT / "tfm_histogram_v1"

for path in (RAW_DIR, LABELS_CSV):
    if not path.exists():
        raise FileNotFoundError(f"Edit the configuration cell; not found: {path}")
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print("Raw EEG:", RAW_DIR)
print("Labels:", LABELS_CSV)
print("Token cache:", TOKEN_CACHE)
print("Results:", RESULTS_DIR)

In [ ]:
# 3) Verify the ZuCo files and label matching before touching the model.
import pandas as pd
from src.zuco_io import inspect_zuco

inspection = pd.DataFrame(inspect_zuco(RAW_DIR, LABELS_CSV))
display(inspection)
print("Subjects:", len(inspection))
print("Matched sentence recordings:", int(inspection.matched_labels.sum()))

In [ ]:
# 4) Shallow-clone official TFM without all weights; download one LFS checkpoint.
# This download happens only in the disposable Colab runtime.
import re, urllib.request

UPSTREAM_URL = "https://github.com/Jathurshan0330/TFM-Tokenizer.git"
UPSTREAM_ROOT = Path("/content/TFM-Tokenizer")
if not UPSTREAM_ROOT.exists():
    environment = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
    run(["git", "clone", "--depth", "1", UPSTREAM_URL, str(UPSTREAM_ROOT)], env=environment)

def lfs_pointer(path):
    try:
        text = path.read_text()
    except (UnicodeDecodeError, OSError):
        return None
    match = re.search(r"^size (\d+)$", text, flags=re.MULTILINE)
    return int(match.group(1)) if text.startswith("version https://git-lfs") and match else None

weight_root = UPSTREAM_ROOT / "pretrained_weigths"  # upstream spelling
candidates = []
for path in weight_root.rglob("*"):
    if path.is_file() and path.suffix.lower() in {".pt", ".pth", ".ckpt"}:
        name = str(path.relative_to(UPSTREAM_ROOT)).lower()
        if any(k in name for k in ("vq", "tokenizer", "tfm_token")) and not any(k in name for k in ("encoder", "classifier", "finetun")):
            candidates.append(path)
if not candidates:
    raise FileNotFoundError("No tokenizer checkpoint candidate found in upstream repository")

candidates.sort(key=lambda p: ("multiple_dataset" not in str(p).lower(), "pretrain" not in p.name.lower(), len(str(p))))
TOKENIZER_CHECKPOINT = candidates[0]
pointer_size = lfs_pointer(TOKENIZER_CHECKPOINT)
if pointer_size is not None:
    relative = TOKENIZER_CHECKPOINT.relative_to(UPSTREAM_ROOT).as_posix()
    print(f"Selected checkpoint: {relative} ({pointer_size / 2**20:.1f} MiB)")
    media_url = f"https://media.githubusercontent.com/media/Jathurshan0330/TFM-Tokenizer/master/{relative}"
    temporary = TOKENIZER_CHECKPOINT.with_suffix(TOKENIZER_CHECKPOINT.suffix + ".download")
    urllib.request.urlretrieve(media_url, temporary)
    if temporary.stat().st_size != pointer_size:
        raise IOError(f"Checkpoint size mismatch: expected {pointer_size}, got {temporary.stat().st_size}")
    temporary.replace(TOKENIZER_CHECKPOINT)
else:
    print("Checkpoint already materialized:", TOKENIZER_CHECKPOINT)

upstream_revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=UPSTREAM_ROOT, text=True).strip()
print("Upstream revision:", upstream_revision)

In [ ]:
# 5) One-recording smoke test. Stop here if shapes or checkpoint loading look wrong.
import torch
from src.config import PreprocessConfig
from src.official_tfm import OfficialTFMTokenizer
from src.preprocess import preprocess_eeg
from src.zuco_io import iter_zuco_recordings

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime → Change runtime type → GPU, then rerun")
preprocess_config = PreprocessConfig()
tokenizer = OfficialTFMTokenizer(UPSTREAM_ROOT, TOKENIZER_CHECKPOINT, device="cuda")
example = next(iter_zuco_recordings(RAW_DIR, LABELS_CSV))
example_eeg = preprocess_eeg(example.eeg, preprocess_config)
example_tokens = tokenizer.tokenize(example_eeg)
print("Subject/sentence/label:", example.subject, example.sentence_id, example.label)
print("Raw -> preprocessed -> tokens:", example.eeg.shape, example_eeg.shape, example_tokens.shape)
print("Token range:", int(example_tokens.min()), int(example_tokens.max()))
print("Checkpoint load report:", tokenizer.load_report)

In [ ]:
# 6) Extract all recordings. This is resumable and writes each result immediately.
from src.extraction import extract_token_cache

extraction_manifest = extract_token_cache(
    raw_dir=RAW_DIR,
    labels_csv=LABELS_CSV,
    cache_dir=TOKEN_CACHE,
    tokenizer=tokenizer,
    preprocess_config=preprocess_config,
    overwrite=False,
)
print(extraction_manifest["report"] | {"failures": len(extraction_manifest["report"]["failures"])})

In [ ]:
# 7) Token-quality diagnostics before classification.
import matplotlib.pyplot as plt
from src.features import build_sentence_histograms

X, y, metadata, diagnostics = build_sentence_histograms(TOKEN_CACHE)
print({k: v for k, v in diagnostics.items() if k != "records"})
display(metadata.head())
display(diagnostics["records"].describe())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
diagnostics["records"]["unique_tokens"].hist(ax=axes[0], bins=30)
axes[0].set(title="Unique tokens per recording", xlabel="count")
diagnostics["records"]["top_token_share"].hist(ax=axes[1], bins=30)
axes[1].set(title="Largest token share", xlabel="fraction")
plt.tight_layout()
plt.show()

In [ ]:
# 8) Sentence-grouped nested CV with aligned, shuffled, and majority setups.
from src.config import EvaluationConfig
from src.evaluation import bootstrap_alignment_delta, evaluate_histograms

evaluation_config = EvaluationConfig()
metrics, predictions, summary = evaluate_histograms(
    X, y, metadata.sentence_id.to_numpy(), RESULTS_DIR, evaluation_config
)
display(summary)
delta = bootstrap_alignment_delta(
    predictions, samples=evaluation_config.bootstrap_samples
)
print("Aligned − shuffled macro-F1:", delta)

In [ ]:
# 9) Apply the predeclared viability rule; do not move to text fusion automatically.
aligned_mean = metrics.loc[metrics.setup == "tfm_histogram", "macro_f1"].mean()
shuffled_mean = metrics.loc[metrics.setup == "tfm_histogram_shuffled", "macro_f1"].mean()
passes = (aligned_mean - shuffled_mean >= 0.015) and (delta["ci_95_low"] > 0)
print(f"Aligned macro-F1:  {aligned_mean:.4f}")
print(f"Shuffled macro-F1: {shuffled_mean:.4f}")
print(f"Observed delta:     {aligned_mean - shuffled_mean:+.4f}")
print("Viability gate:", "PASS — inspect per-seed stability next" if passes else "DO NOT ADVANCE TO TEXT FUSION")
print("Saved results:", RESULTS_DIR)

## After the run

Record the extraction diagnostics, aligned/shuffled scores, bootstrap interval, failures, Colab GPU, and upstream commit in `PROJECT_LOG.md`. A failed viability gate is still a useful transfer result; it should not be tuned away without a new, documented hypothesis.